In [ ]:
import numpy as np
from tabulate import tabulate
from magres.atoms import MagresAtoms
atoms = MagresAtoms.load_magres('/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/CASTEP/D-alanine_295K_278464_invert_cb_rot_17O_opt_magres_new.magres') #latest magres file from 2025

In [12]:
def Rabc(alfa1, beta1, gama1): #Euler Rotation Matrix
    U = np.zeros((3, 3))
    #Changing input angles from degrees to radians
    alfa1 = np.radians(alfa1) 
    beta1 = np.radians(beta1)
    gama1 = np.radians(gama1)
    
    U[0, 0] = np.cos(alfa1)*np.cos(beta1)*np.cos(gama1) - np.sin(alfa1)*np.sin(gama1)
    U[0, 1] = np.sin(alfa1)*np.cos(beta1)*np.cos(gama1) + np.cos(alfa1)*np.sin(gama1)
    U[0, 2] = -np.sin(beta1)*np.cos(gama1)
    U[1, 0] = -np.cos(alfa1)*np.cos(beta1)*np.sin(gama1) - np.sin(alfa1)*np.cos(gama1)
    U[1, 1] = -np.sin(alfa1)*np.cos(beta1)*np.sin(gama1) + np.cos(alfa1)*np.cos(gama1)
    U[1, 2] = np.sin(beta1)*np.sin(gama1)
    U[2, 0] = np.cos(alfa1)*np.sin(beta1)
    U[2, 1] = np.sin(alfa1)*np.sin(beta1)
    U[2, 2] = np.cos(beta1)
    return U

In [13]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    # eigenvectors need to be arranged so that first column for direction cosine matrix is eigenvector for x, second column is for y and third column is for z
    y_dc = sorted_eigenvectors[:,0]
    x_dc = sorted_eigenvectors[:,1]
    z_dc = sorted_eigenvectors[:,2]

    dc = np.stack((x_dc, y_dc, z_dc), axis = 1)


    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, dc,  avg_tensor, eigenvalues, eigenvectors

In [14]:
def get_euler_angles(eigenvectors):
    b = np.degrees(np.arccos(eigenvectors[2,2]))
    a = np.degrees(np.arctan(eigenvectors[2,1]/eigenvectors[2,0]))
    g = np.degrees(np.arctan(-eigenvectors[1,2]/eigenvectors[0,2]))

    return a, b, g
    

In [15]:
for atom in atoms.species('O'):
    print (atom, "sigma:\n",atom.ms.sigma)
    print()

17O1 sigma:
 [[-58.71056973 168.42333854  80.19168888]
 [175.6277732   48.84724812 -29.02649162]
 [ -9.58999475  25.6436215  -52.86906924]]

17O2 sigma:
 [[ -58.71056973 -168.42333854  -80.19168888]
 [-175.6277732    48.84724812  -29.02649162]
 [   9.58999475   25.6436215   -52.86906924]]

17O3 sigma:
 [[-58.71056973 168.42333854 -80.19168888]
 [175.6277732   48.84724812  29.02649162]
 [  9.58999475 -25.6436215  -52.86906924]]

17O4 sigma:
 [[ -58.71056973 -168.42333854   80.19168888]
 [-175.6277732    48.84724812   29.02649162]
 [  -9.58999475  -25.6436215   -52.86906924]]

17O5 sigma:
 [[ -47.51972884  228.54720094  -55.0666635 ]
 [ 197.59879891  101.82413744   88.59783075]
 [ -16.91499843   50.38859125 -180.28276326]]

17O6 sigma:
 [[ -47.51972884 -228.54720094   55.0666635 ]
 [-197.59879891  101.82413744   88.59783075]
 [  16.91499843   50.38859125 -180.28276326]]

17O7 sigma:
 [[ -47.51972884  228.54720094   55.0666635 ]
 [ 197.59879891  101.82413744  -88.59783075]
 [  16.91499843

In [16]:
for atom in atoms.species('O'):
    print (atom, "sigma:\n",atom.efg.Cq)
    print()

17O1 sigma:
 6.353653917778783

17O2 sigma:
 6.353653917778768

17O3 sigma:
 6.353653917778764

17O4 sigma:
 6.353653917778741

17O5 sigma:
 8.336943250025397

17O6 sigma:
 8.336943250025401

17O7 sigma:
 8.336943250025469

17O8 sigma:
 8.336943250025486



In [17]:
# using values from latest magres file
Cs = np.zeros((3, 3))                                # CS symmetric (l = 0 + 2) Tensor from updated_magres
CS_anti = np.zeros((3,3))                           # CS antisymmetric ( l = 1)
CS_iso = np.zeros((3,3))                            # CS isotropic  (l = 0)
CS_total = np.zeros((3,3))  # CS total shielding tensor ( l = 0 + 1 + 2) Tensor from magres

atom_label = 0
CS_total[:,:] = atoms.species('O').ms.sigma[atom_label]

iso = np.mean([CS_total[0,0], CS_total[1,1], CS_total[2,2]]) # isotropic chemical shielding (l = 0)

CS_iso[0,0] = CS_iso[1,1] = CS_iso[2,2] = iso

Cs[0,0] = CS_total[0,0]; Cs[0,1] = (CS_total[0,1] + CS_total[1,0] )/2; Cs[0,2] = (CS_total[0,2] + CS_total[2,0])/2;
Cs[1,0] = Cs[0,1];      Cs[1,1] = CS_total[1,1];                      Cs[1,2] = (CS_total[1,2] + CS_total[2,1])/2;
Cs[2,0] = Cs[0,2];      Cs[2,1] = Cs[1,2];                           Cs[2,2] = CS_total[2,2];

CS_anti[0,1] = (CS_total[0,1] - CS_total[1,0])/2; CS_anti[0,2] = (CS_total[0,2] - CS_total[2,0])/2; 
CS_anti[1,0] = -CS_anti[0,1]; CS_anti[1,2] = (CS_total[1,2] - CS_total[2,1])/2;
CS_anti[2,0] = -CS_anti[0,2];      CS_anti[2,1] = -CS_anti[1,2]; 


efg = np.zeros((3, 3 ))                             # EFG Tensor from magres (in a.u.)

efg[:,:] = atoms.species('O')[atom_label].efg.V


# Convert a.u. units to MHz
# Q tensor elements (MHz) = efg tensor (a.u.)* Q (barn) * 234.9647 
# Q = 0.04059 barn https://www-nds.iaea.org/publications/indc/indc-nds-0650.pdf
Q = -0.0256 #electric quadrupole moment for O17 in barn
V = efg*Q*234.9647

print('\nQ tensor:\n', np.round(V,3))
print('\nCS Tensor:\n',np.round(CS_total, 3))
print('\nCS isotropic Tensor:\n',np.round(CS_iso, 3))
print('\nCS symmetric Tensor:\n',np.round(Cs,3))
print('\nCS antisymmetric Tensor:\n',np.round(CS_anti,3))



Q tensor:
 [[ 0.875 -0.844 -4.509]
 [-0.844 -0.447  3.84 ]
 [-4.509  3.84  -0.428]]

CS Tensor:
 [[-58.711 168.423  80.192]
 [175.628  48.847 -29.026]
 [ -9.59   25.644 -52.869]]

CS isotropic Tensor:
 [[-20.911   0.      0.   ]
 [  0.    -20.911   0.   ]
 [  0.      0.    -20.911]]

CS symmetric Tensor:
 [[-58.711 172.026  35.301]
 [172.026  48.847  -1.691]
 [ 35.301  -1.691 -52.869]]

CS antisymmetric Tensor:
 [[  0.     -3.602  44.891]
 [  3.602   0.    -27.335]
 [-44.891  27.335   0.   ]]


In [18]:
print("For EFG tensor")
sorted_eigenvalues_efg, dc_efg, quad_avg, eigenvalues_efg, eigenvectors_efg = sort_eigenvalues(V)
print('==================================\n')
print("For CS tensor")
sorted_eigenvalues_cs, dc_cs, cs_avg, eigenvalues_cs, eigenvectors_cs = sort_eigenvalues(Cs)

For EFG tensor
 Unsorted Eigenvalues:
 [ 6.35861941 -0.7218601  -5.63675931] 

 Unsorted Eigenvectors:
 [[-0.60845048  0.6491058   0.45656289]
 [ 0.44586389  0.75553569 -0.47997001]
 [ 0.65650088  0.08847307  0.74911889]] 

Sorted Eigenvalues: 
 [-0.7218601  -5.63675931  6.35861941] 

Sorted Eigenvectors: 
 [[ 0.6491058   0.45656289 -0.60845048]
 [ 0.75553569 -0.47997001  0.44586389]
 [ 0.08847307  0.74911889  0.65650088]] 


For CS tensor
 Unsorted Eigenvalues:
 [ 176.98374716 -191.47010786  -48.24603014] 

 Unsorted Eigenvectors:
 [[ 0.59571124  0.79443357  0.11833606]
 [ 0.79862295 -0.57014977 -0.19269308]
 [ 0.08561258 -0.20929532  0.97409751]] 

Sorted Eigenvalues: 
 [ -48.24603014 -191.47010786  176.98374716] 

Sorted Eigenvectors: 
 [[ 0.11833606  0.79443357  0.59571124]
 [-0.19269308 -0.57014977  0.79862295]
 [ 0.97409751 -0.20929532  0.08561258]] 



In [19]:

#Calculate Quadrupolar Tensor in PAS

Vyy = sorted_eigenvalues_efg[0]
Vxx = sorted_eigenvalues_efg[1]
Vzz = sorted_eigenvalues_efg[2]

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================')

#Calculate CSA Tensor in PAS

Csyy = sorted_eigenvalues_cs[0] 
Csxx = sorted_eigenvalues_cs[1]  
Cszz = sorted_eigenvalues_cs[2]



print('CSA Tensor Components δyy, δxx, δzz: \n', Csyy, Csxx, Cszz)

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 -0.7218601011987128 -5.636759311129465 6.358619412328167
CSA Tensor Components δyy, δxx, δzz: 
 -48.2460301421398 -191.47010785989684 176.98374716011324


In [20]:
iso_cs = (Csxx + Csyy + Cszz)/3

csa = Cszz - iso_cs
etas = (Csyy - Csxx)/csa

#for Quadrupolar
CQ_fit = Vzz

etaq = (Vyy - Vxx)/Vzz

table = [['CQ (MHz)', CQ_fit], ['etaq', etaq ], ['iso_cs (ppm)',iso_cs ],['csa (ppm)', csa],  ['etas', etas]  ]
print(tabulate(table, headers=['Qauntity', 'Value']))

Qauntity           Value
------------  ----------
CQ (MHz)        6.35862
etaq            0.772951
iso_cs (ppm)  -20.9108
csa (ppm)     197.895
etas            0.723739


In [21]:
# Calculation for efg tensor
print('Direction cosine efg:\n')
print(dc_efg, '\n')
a_efg, b_efg, g_efg = get_euler_angles(dc_efg)

print("Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:")
print(a_efg, b_efg, g_efg, '\n')

print('=========================')
print('Direction cosine csa: \n')
print(dc_cs, '\n')
a_cs, b_cs, g_cs = get_euler_angles(dc_cs)

print("Calculated Euler angles (degrees) CSA PAS --> Crystal:")
print(a_cs, b_cs, g_cs, '\n')

Direction cosine efg:

[[ 0.45656289  0.6491058  -0.60845048]
 [-0.47997001  0.75553569  0.44586389]
 [ 0.74911889  0.08847307  0.65650088]] 

Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:
6.735593702761211 48.96644734704963 36.23343336461915 

Direction cosine csa: 

[[ 0.79443357  0.11833606  0.59571124]
 [-0.57014977 -0.19269308  0.79862295]
 [-0.20929532  0.97409751  0.08561258]] 

Calculated Euler angles (degrees) CSA PAS --> Crystal:
-77.87374577112404 85.08874853543045 -53.27989533220332 



In [22]:
# Euler Matrix to relate Quadrupolar and CSA tensor

CSA_Q = np.matmul(np.linalg.inv(dc_efg), (dc_cs))

psi, chi, xi = get_euler_angles(CSA_Q)

print("Calculated Euler angles (degrees) PAS CSA --> Quadrupole:")
print('psi:', psi, 'chi:', chi, 'xi:', xi, '\n')

Calculated Euler angles (degrees) PAS CSA --> Quadrupole:
psi: -28.827756555857224 chi: 87.14428000670503 xi: 87.29118622035519 



**Rotation of tensors Crystal--> Tenon Frame**

In [23]:
#Euler angles Crystal--> Tenon Frame for LHQ
alpha = 280
beta = 45
gamma = 180
U  = Rabc(alpha, beta, gamma)
Cs_tenon = np.matmul(np.matmul(np.linalg.inv(U), Cs), U)

V_tenon = np.matmul(np.matmul(np.linalg.inv(U), V), U)

print('CSA Tensor in Crystal Frame: \n', Cs) #from magres
print('CSA Tensor in Tenon Frame: \n', Cs_tenon)

print('==========================')
print('Quad Tensor in Crystal Frame: \n', V)  #from magres
print('Quad Tensor in Tenon Frame: \n', V_tenon)


CSA Tensor in Crystal Frame: 
 [[-58.71056973 172.02555587  35.30084706]
 [172.02555587  48.84724812  -1.69143506]
 [ 35.30084706  -1.69143506 -52.86906924]]
CSA Tensor in Tenon Frame: 
 [[  86.64014812  -91.49772436 -118.10740768]
 [ -91.49772436 -128.88356654  -23.79133012]
 [-118.10740768  -23.79133012  -20.48897242]]
Quad Tensor in Crystal Frame: 
 [[ 0.87490914 -0.84380188 -4.5092786 ]
 [-0.84380188 -0.44655021  3.83970206]
 [-4.5092786   3.83970206 -0.42835892]]
Quad Tensor in Tenon Frame: 
 [[-1.42306202  2.22633626 -2.19939282]
 [ 2.22633626  5.70906551  0.27387424]
 [-2.19939282  0.27387424 -4.28600349]]
